# 18 — Can the point estimate get better? Judged by the NASA score first

**Why this notebook exists.** Before this notebook, the deployed FD001 point estimate was the RSF alone: test RMSE 15.6, NASA 4.05 per engine. RMSE is the number C-MAPSS papers compare, but this project's primary metric is the asymmetric NASA score, because a late prediction means an engine fails in service. So the question is not "can RMSE go down?" but **"can the point estimate improve on the NASA score without getting worse elsewhere?"**, the same priority the retraining gate enforces.

**Part 1 — where the error comes from** (deployed model, official test set, descriptive only).

**Part 2 — candidates, fixed before running:**

| Candidate | What changes |
|---|---|
| RSF, cap 125 | the deployed model (baseline) |
| RSF, cap 115 / 135 / 145 | the RUL cap, a convention never tuned in this project |
| Gradient boosting, cap 125 | a regressor on the capped target (`HistGradientBoostingRegressor`, 300 iterations, learning rate 0.05, not tuned) — reference only |
| Blend, cap 125 | the average of RSF and gradient boosting |

All use the deployed features, `min_samples_leaf=5` and `row_stride=3`.

**Protocol:** the project's cross-validation (notebooks 08, 11): 5 folds by engine, each held-out engine scored at random points between 30% and 90% of its life, here 5 draws per engine instead of one to reduce noise.

**Selection rule:** a candidate replaces the baseline only if (a) its mean NASA score per engine is at least 5% better, (b) it beats the baseline in at least 4 of 5 folds, (c) it is not worse on held-out points with at most 150 cycles left (where maintenance decisions are made, and the range of the official test set), and (d) its RMSE is not more than 5% worse. Among candidates that pass, the lowest NASA score wins. The winner, if any, is checked once on the official test set and then sent through the retraining gate as a real challenger.

**Test-set rule, also fixed in advance:** the winner is worth deploying only if its official-test NASA score per engine is lower than the deployed model's *and* it then passes the retraining gate.

**Disclosure.** The first run's selection code left gradient boosting alone in the candidate list, against the table above, and picked it. The code was corrected to match the written plan, which was set before the run; gradient boosting's numbers stay in every table as a reference.

**Why rule (c).** The NASA penalty for early predictions grows exponentially, and CV scores engines up to ~250 cycles from failure, further than any official test engine. Without (c), a larger cap could win simply by predicting more life for very young engines, which is not where the model is used.

In [1]:
import sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold
from sksurv.metrics import concordance_index_censored

sys.path.append(str(Path("..") / "src"))
from turbofan_rul import cross_validation, data_loading, evaluation, features, inference, labeling, models, plotting

plotting.style()
DATA_DIR = Path("..") / "data" / "raw"
PROCESSED_DIR = Path("..") / "data" / "processed"
MODELS_DIR = Path("..") / "models"
CAPS = [115, 125, 135, 145]
LEAF, STRIDE, N_DRAWS = 5, 3, 5
RECOMPUTE = False   # True re-fits every fold model (~25 minutes)

cmapss = data_loading.load_subset("FD001", data_dir=DATA_DIR)
kept = features.load_kept_sensors(PROCESSED_DIR / "fd001_sensor_selection.json")
META = {"unit_number", "time_cycles", "RUL", "duration", "event"}

def metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return {"NASA / engine": evaluation.nasa_score_mean(y, p), "RMSE": evaluation.rmse(y, p),
            "MAE": evaluation.mae(y, p), "late rate": evaluation.late_prediction_rate(y, p),
            "C-index": concordance_index_censored(np.ones(len(y), bool), y, -p)[0]}

## 1. Where does the deployed model's error come from?

In [2]:
bundle = inference.load_deployment_bundle(MODELS_DIR / "fd001")
true_rul = cmapss.rul.set_index("unit_number")["RUL"]
rows = []
for unit, engine in cmapss.test.groupby("unit_number"):
    r = inference.predict(bundle, engine.sort_values("time_cycles")[["time_cycles"] + bundle.raw_passthrough_cols])
    # The RSF component, whatever the bundle now deploys: this notebook compares against RSF alone.
    rows.append({"unit": unit, "true": true_rul.loc[unit],
                 "pred": r["point_estimate_components"]["random_survival_forest"]})
test_pred = pd.DataFrame(rows)
test_pred["sq_err"] = (test_pred.pred - test_pred.true) ** 2
test_pred["band"] = pd.cut(test_pred.true, [0, 25, 50, 75, 100, 125, np.inf],
                           labels=["0-25", "26-50", "51-75", "76-100", "101-125", ">125"])
y, p = test_pred.true.values, test_pred.pred.values
penalty = np.where(p - y > 0, np.exp((p - y) / 10) - 1, np.exp(-(p - y) / 13) - 1)
test_pred["nasa"] = penalty
breakdown = test_pred.groupby("band", observed=True).agg(
    engines=("unit", "size"), RMSE=("sq_err", lambda s: np.sqrt(s.mean())), mean_error=("pred", "mean"),
    share_of_squared_error=("sq_err", lambda s: s.sum() / test_pred.sq_err.sum()),
    share_of_NASA=("nasa", lambda s: s.sum() / test_pred.nasa.sum()))
breakdown["mean_error"] = test_pred.groupby("band", observed=True).apply(lambda g: (g.pred - g.true).mean(), include_groups=False)
print("RSF alone (the deployed point estimate before this notebook), official test set:", {k: round(v, 3) for k, v in metrics(y, p).items()})
print(f"RMSE without the {int((y > 125).sum())} engines above the cap: {evaluation.rmse(y[y <= 125], p[y <= 125]):.2f}")
breakdown.round(3)

RSF alone (the deployed point estimate before this notebook), official test set: {'NASA / engine': 4.049, 'RMSE': 15.557, 'MAE': 11.332, 'late rate': 0.56, 'C-index': np.float64(0.864)}
RMSE without the 11 engines above the cap: 14.14


,engines,RMSE,mean_error,share_of_squared_error,share_of_NASA
band,,,,,
0-25,19,5.963,2.395,0.028,0.032
26-50,14,13.054,6.079,0.099,0.132
51-75,10,20.418,14.450,0.172,0.254
76-100,24,17.015,7.900,0.287,0.314
101-125,22,12.824,-3.218,0.149,0.115
>125,11,24.130,-23.264,0.265,0.152


## 2. Cross-validated candidates

In [3]:
oof_path = PROCESSED_DIR / "point_candidates_oof_fd001.csv"
if oof_path.exists() and not RECOMPUTE:
    oof = pd.read_csv(oof_path)
else:
    units = cmapss.train.unit_number.unique()
    featured_all = features.build_feature_matrix(cmapss.train, kept_sensors=kept)
    out = []
    for fold, (tr_pos, te_pos) in enumerate(GroupKFold(n_splits=5).split(units, groups=units)):
        t0 = time.time()
        tr_raw = cmapss.train[cmapss.train.unit_number.isin(units[tr_pos])]
        te_raw = cmapss.train[cmapss.train.unit_number.isin(units[te_pos])]
        te_feat = featured_all[featured_all.unit_number.isin(units[te_pos])]
        evals = pd.concat([cross_validation.build_truncated_eval_frame(
            te_feat, cross_validation.sample_truncation_points(te_raw, random_state=42 + k)).assign(draw=k)
            for k in range(N_DRAWS)], ignore_index=True)
        frame = {"fold": fold, "unit_number": evals.unit_number.values, "draw": evals.draw.values,
                 "true_rul": evals.true_rul.values}
        for cap in CAPS:
            train = features.build_feature_matrix(labeling.build_train_survival_frame(tr_raw, cap=cap), kept_sensors=kept)
            cols = [c for c in train.columns if c not in META]
            rsf = models.fit_random_survival_forest(train, cols, row_stride=STRIDE, min_samples_leaf=LEAF, random_state=42)
            frame[f"rsf_{cap}"] = models.predict_rul_from_survival(rsf, evals[cols]).pred_rul.values
            if cap == 125:
                sub = train.iloc[::STRIDE]
                gbm = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=0)
                gbm.fit(sub[cols], sub["duration"])
                frame["gbm_125"] = np.clip(gbm.predict(evals[cols]), 0, None)
        out.append(pd.DataFrame(frame))
        print(f"fold {fold}: {len(evals)} held-out points, {time.time() - t0:.0f}s")
    oof = pd.concat(out, ignore_index=True)
    oof["blend_125"] = (oof.rsf_125 + oof.gbm_125) / 2
    oof.to_csv(oof_path, index=False)

CANDIDATES = {"RSF cap 125 (deployed)": "rsf_125", "RSF cap 115": "rsf_115", "RSF cap 135": "rsf_135",
              "RSF cap 145": "rsf_145", "Gradient boosting cap 125": "gbm_125", "Blend RSF + GB, cap 125": "blend_125"}
print(len(oof), "held-out points;", oof.groupby("fold").size().to_dict())

500 held-out points; {0: 100, 1: 100, 2: 100, 3: 100, 4: 100}


In [4]:
def summarise(frame):
    rows = {}
    for name, col in CANDIDATES.items():
        per_fold = frame.groupby("fold").apply(lambda g: evaluation.nasa_score_mean(g.true_rul, g[col]), include_groups=False)
        near = frame[frame.true_rul <= 150]
        rows[name] = {**metrics(frame.true_rul, frame[col]),
                      "NASA per fold": per_fold.round(2).tolist(),
                      "NASA / engine, true RUL <= 150": evaluation.nasa_score_mean(near.true_rul, near[col]),
                      "RMSE, true RUL <= 150": evaluation.rmse(near.true_rul, near[col])}
    return pd.DataFrame(rows).T

cv = summarise(oof)
display(cv.drop(columns="NASA per fold").astype(float).round(3))
display(cv[["NASA per fold"]])

,NASA / engine,RMSE,MAE,late rate,C-index,"NASA / engine, true RUL <= 150","RMSE, true RUL <= 150"
RSF cap 125 (deployed),78.830,23.584,16.760,0.546,0.840,6.764,18.316
RSF cap 115,106.802,23.892,16.468,0.524,0.843,5.453,17.782
RSF cap 135,66.499,23.523,16.940,0.582,0.841,8.239,18.924
RSF cap 145,60.399,23.995,17.417,0.588,0.837,10.138,19.864
Gradient boosting cap 125,62.470,21.901,14.500,0.562,0.863,5.612,16.199
"Blend RSF + GB, cap 125",68.395,22.291,15.266,0.586,0.856,5.497,16.660


,NASA per fold
RSF cap 125 (deployed),"[8.46, 187.75, 10.6, 88.37, 98.97]"
RSF cap 115,"[7.12, 276.07, 11.2, 101.19, 138.44]"
RSF cap 135,"[9.96, 144.13, 11.68, 106.46, 60.26]"
RSF cap 145,"[11.55, 120.99, 14.1, 84.51, 70.85]"
Gradient boosting cap 125,"[6.28, 142.94, 7.19, 51.52, 104.42]"
"Blend RSF + GB, cap 125","[6.48, 163.12, 7.73, 63.76, 100.88]"


In [5]:
base = cv.loc["RSF cap 125 (deployed)"]
base_folds = np.array(base["NASA per fold"])
verdicts = {}
# Gradient boosting alone is a reference, not a candidate (see the table above):
# the deployed point estimate stays survival-based, alone or blended.
REFERENCE_ONLY = {"RSF cap 125 (deployed)", "Gradient boosting cap 125"}
for name in CANDIDATES:
    if name in REFERENCE_ONLY:
        continue
    r = cv.loc[name]
    checks = {
        "(a) NASA >=5% better": r["NASA / engine"] <= 0.95 * base["NASA / engine"],
        "(b) wins >=4 of 5 folds": int((np.array(r["NASA per fold"]) < base_folds).sum()) >= 4,
        "(c) not worse at true RUL <= 150": r["NASA / engine, true RUL <= 150"] <= base["NASA / engine, true RUL <= 150"],
        "(d) RMSE at most 5% worse": r["RMSE"] <= 1.05 * base["RMSE"],
    }
    verdicts[name] = {**checks, "passes": all(checks.values())}
verdicts = pd.DataFrame(verdicts).T
display(verdicts)
passing = [n for n in verdicts.index if verdicts.loc[n, "passes"]]
winner = min(passing, key=lambda n: cv.loc[n, "NASA / engine"]) if passing else None
print("Selected on cross-validation:", winner or "none -- the deployed model stays")

,(a) NASA >=5% better,(b) wins >=4 of 5 folds,(c) not worse at true RUL <= 150,(d) RMSE at most 5% worse,passes
RSF cap 115,False,False,True,True,False
RSF cap 135,True,False,False,True,False
RSF cap 145,True,False,False,True,False
"Blend RSF + GB, cap 125",True,True,True,True,True


Selected on cross-validation: Blend RSF + GB, cap 125


## 3. The winner on the official test set (once)

In [6]:
assert winner == "Blend RSF + GB, cap 125", winner
train125 = features.build_feature_matrix(labeling.build_train_survival_frame(cmapss.train, cap=125), kept_sensors=kept)
sub = train125.iloc[::STRIDE]
gbm_full = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=0).fit(
    sub[bundle.full_feature_cols], sub["duration"])
test_feat = features.build_feature_matrix(cmapss.test, kept_sensors=kept).groupby("unit_number").tail(1)
test_feat = test_feat.set_index("unit_number").loc[test_pred.unit]
gb_test = np.clip(gbm_full.predict(test_feat[bundle.full_feature_cols]), 0, None)
test_pred["gbm"] = gb_test
test_pred["blend"] = (test_pred.pred + test_pred.gbm) / 2
confirm = pd.DataFrame({name: metrics(test_pred.true, test_pred[col]) for name, col in
                        [("RSF alone", "pred"), ("Blend RSF + GB (selected)", "blend"),
                         ("Gradient boosting (reference)", "gbm")]}).T
display(confirm.astype(float).round(3))
better = confirm.loc["Blend RSF + GB (selected)", "NASA / engine"] < confirm.loc["RSF alone", "NASA / engine"]
print("Test-set rule -- blend NASA lower than deployed:", "PASSED" if better else "FAILED")

,NASA / engine,RMSE,MAE,late rate,C-index
RSF alone,4.049,15.557,11.332,0.56,0.864
Blend RSF + GB (selected),3.120,14.241,10.436,0.62,0.877
Gradient boosting (reference),3.226,13.898,10.114,0.60,0.876


Test-set rule -- blend NASA lower than deployed: PASSED


## 3b. How late are the late errors?

The blend is late on more engines than RSF alone (section 3), which looks like a step backwards for a safety-relevant model. But the late-prediction rate counts a 1-cycle overshoot the same as a 30-cycle one. This section measures how large the late errors are, overall and near failure, on the same official test predictions. It is descriptive: it was added after the selection and changes nothing about it.

The deployed safety decision runs off the interval's lower bound, which the blend doesn't change (zero dangerous misses on this test set). This table is about the point estimate only.

In [7]:
def late_severity(y, p, near=25):
    y, p = np.asarray(y, float), np.asarray(p, float)
    d = p - y
    nf = y <= near
    return {"late (any)": int((d > 0).sum()), "mean late error": d[d > 0].mean(),
            "late by >20": int((d > 20).sum()), "late by >30": int((d > 30).sum()), "worst late": d.max(),
            f"near failure (<= {near}): late by >10": int((d[nf] > 10).sum()),
            f"near failure (<= {near}): worst late": d[nf].max(),
            "true <= 50: late by >20": int((d[y <= 50] > 20).sum())}

severity = pd.DataFrame({name: late_severity(test_pred.true, test_pred[col]) for name, col in
                         [("RSF alone", "pred"), ("Blend RSF + GB (selected)", "blend"),
                          ("Gradient boosting (reference)", "gbm")]}).T
display(severity.astype(float).round(2))
r, b = severity.loc["RSF alone"], severity.loc["Blend RSF + GB (selected)"]
print(f"Late at all: RSF {int(r['late (any)'])}, blend {int(b['late (any)'])} of {len(test_pred)} engines; "
      f"mean size of a late error: RSF {r['mean late error']:.1f}, blend {b['mean late error']:.1f} cycles")
fewer = bool(b["late by >20"] <= r["late by >20"] and b["late by >30"] <= r["late by >30"])
print("Blend has no more large late errors (>20 and >30 cycles) than RSF alone:", fewer)

,late (any),mean late error,late by >20,late by >30,worst late,near failure (<= 25): late by >10,near failure (<= 25): worst late,true <= 50: late by >20
RSF alone,56.0,11.35,10.0,5.0,38.50,2.0,21.90,3.0
Blend RSF + GB (selected),62.0,9.24,7.0,2.0,36.65,1.0,16.39,1.0
Gradient boosting (reference),60.0,8.98,5.0,3.0,38.40,1.0,10.88,0.0


Late at all: RSF 56, blend 62 of 100 engines; mean size of a late error: RSF 11.3, blend 9.2 cycles
Blend has no more large late errors (>20 and >30 cycles) than RSF alone: True


## 4. Through the retraining gate, and findings

**The gate.** The blend was exported as a real challenger (`python -m turbofan_rul.export_deployment_artifacts --subset FD001 --point-model blend --models-dir models_challenger`) and judged against the RSF-only champion by `gate.py`, on the four rules fixed in Phase 26:

| Check | Champion (RSF) | Challenger (blend) | Rule | Result |
|---|---|---|---|---|
| NASA score per engine | 4.049 | 3.119 | ≤ 4.251 | pass |
| Test C-index | 0.864 | 0.877 | ≥ 0.854 | pass |
| Interval coverage | 0.97 | 0.97 | ≥ 0.85 | pass |
| Coverage near failure | 1.00 | 1.00 | ≥ 0.95 | pass |

**Promoted.** FD001's deployed recipe is now the blend (`export_deployment_artifacts.POINT_MODEL_BY_SUBSET`); the interval is unchanged, since it never uses the point estimate. The blend adds a 1.5 MB model.

**Findings.**
1. **The error is concentrated, not spread.** The 11 test engines with more than 125 cycles left are 11% of engines but about a quarter of the squared error: RSF can't predict past the cap, so it under-predicts them by ~23 cycles. Without them, RSF's RMSE is 14.1, not 15.6.
2. **Rule (c) mattered.** Caps of 135 and 145 "won" on cross-validated NASA overall, but only by predicting more life for very young engines; on points with at most 150 cycles left they were worse, so they were rejected. Cap 115 was worse overall.
3. **Blending with a regressor helped on every metric.** Cross-validated NASA 68.4 vs 78.8, 4 of 5 folds; official test NASA 3.12 vs 4.05 per engine (−23%), RMSE 14.2 vs 15.6, C-index 0.877 vs 0.864. Its late rate is higher (62% vs 56%), but its late errors are smaller (section 3b), which is what the NASA score rewards.
4. **Gradient boosting alone was close to the blend on the test set** (NASA 3.23 vs 3.12, RMSE 13.9 vs 14.2). It stayed a reference, as the plan said: the deployed point estimate remains survival-based, with the forest's survival curve one of the two members, and the blend was slightly better on the ≤150-cycle range in cross-validation.

**Scope.** FD001 only. FD004 keeps the RSF-only point estimate; the same experiment on FD004 would be the next step. The test set had been used before, so the test confirmation is a check, not an independent estimate; the cross-validated selection is the main evidence. Gradient boosting's fit shifts slightly with numpy/pandas versions (another environment gave test NASA 3.07 for the blend); the RSF reproduces exactly, and the selection and gate decisions are the same in both.